In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
import os

# DIRECT LSTM — GENERATION PIPELINE

device_torch = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQ_LEN = 20

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import sumolib

# SUMO MODULE

sumo_home = os.path.expanduser('~/sumo')
sys.path.append(os.path.join(sumo_home, 'tools'))
os.environ['SUMO_HOME'] = sumo_home

SUMO_DIR = 'D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1'
net_file = os.path.join(SUMO_DIR, 'osm1.net.xml')

print("Loading SUMO network...")
net = sumolib.net.readNet(net_file)
print(f"  Network loaded: {len(net.getEdges())} edges")

Loading SUMO network...
  Network loaded: 6055 edges


In [3]:
BASE_INPUTS = [
    'hour', 'day_of_week',
    'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4',
    'direction_uplink',
    'measured_qos_delay',
    'measurement', 'operator'
]

ALL_TARGETS = [
    'Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG', 'Altitude',
    'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed',
    'Traffic Jam Factor', 'Traffic Distance',
    'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
    'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
    'PCell_Downlink_frequency', 'PCell_Band_Indicator',
    'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
    'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size',
    'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
    'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)',
    'datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate',
    'ping_ms'
]

ALL_AUTO_COLS = ALL_TARGETS.copy()

SNAP_RULES = {
    'PCell_Downlink_frequency': [125.0, 475.0, 1300.0, 1801.0, 2850.0, 3050.0, 3749.0, 9460.0],
    'PCell_freq_MHz': [700.0, 900.0, 1800.0, 2000.0, 2100.0, 2600.0],
    'PCell_Band_Indicator': [1.0, 3.0, 7.0, 8.0, 28.0],
    'PCell_Downlink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Uplink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Downlink_Average_MCS': list(range(0, 30)),
}

LOG_FEATURES = [
    'datarate', 'target_datarate',
    'PCell_Downlink_TB_Size', 'PCell_Uplink_TB_Size',
    'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
    'PCell_Downlink_Num_RBs', 'PCell_Uplink_Num_RBs',
    'ping_ms', 'Pos in Ref Round', 'Traffic Distance',
    'PCell_Uplink_Tx_Power_(dBm)',
]

def snap_to_nearest(value, valid_set):
    valid_arr = np.array(valid_set)
    return valid_arr[np.argmin(np.abs(valid_arr - value))]

In [4]:
class DirectLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim,
                            num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0,
                            batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

In [5]:

MODEL_DIR = 'research\generation_output'

print("Loading Direct LSTM model and scalers...")
scalers = pickle.load(open(os.path.join(MODEL_DIR, 'direct_multioutput_scalers.pkl'), 'rb'))
input_scaler = scalers['input_scaler']
target_scaler = scalers['target_scaler']



model = DirectLSTM(input_dim=47, hidden_dim=256, output_dim=37, num_layers=2, dropout=0.20)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'direct_lstm_model.pt'), map_location=device_torch))
model.to(device_torch)
model.eval()
print(f"  Model loaded: {sum(p.numel() for p in model.parameters())} parameters")

# Load dataset
print("Loading dataset...")
df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')
df = df.sort_values('timestamp').reset_index(drop=True)
df['sin_COG'] = np.sin(np.radians(df['COG']))
df['cos_COG'] = np.cos(np.radians(df['COG']))
df['jitter_log'] = np.log1p(df['jitter'])
print(f"  Dataset loaded: {len(df)} rows")

Loading Direct LSTM model and scalers...
  Model loaded: 876325 parameters
Loading dataset...
  Dataset loaded: 204942 rows


In [6]:

# ROAD POSITION
def get_road_position(timestamp, last_row, net=net):
   
    ts = pd.Timestamp(timestamp)
    if ts.tzinfo is None:
        ts = ts.tz_localize('Europe/Berlin')
    
    last_ts = pd.Timestamp(last_row['timestamp'])
    time_diff = (ts - last_ts).total_seconds()
    
    if time_diff <= 0:
        return None
    
    speed_ms = max(last_row['speed_kmh'] / 3.6, 1.0)
    target_dist = speed_ms * time_diff
    last_cog = last_row['COG']
    
    # Find current edge and offset
    x, y = net.convertLonLat2XY(last_row['Longitude'], last_row['Latitude'])
    nearby = net.getNeighboringEdges(x, y, r=100)
    valid = [(e, d) for e, d in nearby if not e.getID().startswith(':')]
    if not valid:
        valid = nearby
    if not valid:
        return None
    
    edge, _ = sorted(valid, key=lambda e: e[1])[0]
    
    # Find offset on edge
    shape = edge.getShape()
    min_d = float('inf')
    car_offset = 0
    for i in range(len(shape) - 1):
        x1, y1 = shape[i]
        x2, y2 = shape[i + 1]
        dx, dy = x2 - x1, y2 - y1
        seg_len = np.sqrt(dx**2 + dy**2)
        if seg_len > 0:
            t = max(0, min(1, ((x - x1)*dx + (y - y1)*dy) / seg_len**2))
            px, py = x1 + t*dx, y1 + t*dy
            d = np.sqrt((x - px)**2 + (y - py)**2)
            if d < min_d:
                min_d = d
                car_offset = sum(
                    np.sqrt((shape[k+1][0]-shape[k][0])**2 + (shape[k+1][1]-shape[k][1])**2)
                    for k in range(i)
                ) + t * seg_len
    
    # Walk along road
    dist_remaining = target_dist
    offset = car_offset
    current_edge = edge
    
    while dist_remaining > 0:
        remaining_on_edge = current_edge.getLength() - offset
        
        if dist_remaining <= remaining_on_edge:
            final_offset = offset + dist_remaining
            pos = sumolib.geomhelper.positionAtShapeOffset(current_edge.getShape(), final_offset)
            lon, lat = net.convertXY2LonLat(pos[0], pos[1])
            return {
                'lat': lat, 'lon': lon,
                'edge': current_edge.getName() or current_edge.getID(),
                'speed_limit': current_edge.getSpeed() * 3.6,
                'dist_from_car': round(target_dist, 1),
                'time_diff': time_diff,
                'timestamp': str(ts),
            }
        
        dist_remaining -= remaining_on_edge
        offset = 0
        
        outgoing = current_edge.getOutgoing()
        valid_out = [e for e in outgoing if not e.getID().startswith(':')]
        if not valid_out:
            valid_out = list(outgoing) if outgoing else []
        
        if not valid_out:
            pos = sumolib.geomhelper.positionAtShapeOffset(
                current_edge.getShape(), current_edge.getLength())
            lon, lat = net.convertXY2LonLat(pos[0], pos[1])
            return {
                'lat': lat, 'lon': lon,
                'edge': current_edge.getName() or current_edge.getID(),
                'speed_limit': current_edge.getSpeed() * 3.6,
                'dist_from_car': round(target_dist, 1),
                'time_diff': time_diff,
                'timestamp': str(ts),
            }
        
        best_edge = None
        best_diff = 999
        for e in valid_out:
            e_shape = e.getShape()
            if len(e_shape) >= 2:
                heading = np.degrees(np.arctan2(
                    e_shape[1][0] - e_shape[0][0],
                    e_shape[1][1] - e_shape[0][1]
                )) % 360
                diff = abs(last_cog - heading)
                diff = min(diff, 360 - diff)
                if diff < best_diff:
                    best_diff = diff
                    best_edge = e
        
        current_edge = best_edge if best_edge else valid_out[0]
    
    return None


In [7]:
def generate_direct_lstm(device_id, start_timestamp, n_steps=5, mode='with_gps',
                         lat=None, lon=None, positions=None):
    """
    Parameters:
        device_id:       'pc1', 'pc2', 'pc3', or 'pc4'
        start_timestamp: pd.Timestamp
        n_steps:         number of timesteps to generate (1-10 recommended)
        mode:            'with_gps' (lat/lon provided)
        lat:             latitude (required if mode='with_gps')
        lon:             longitude (required if mode='with_gps')
        positions:       list of dicts with 'lat','lon' per step
    """
    start_ts = pd.Timestamp(start_timestamp)
    if start_ts.tzinfo is None:
        start_ts = start_ts.tz_localize('Europe/Berlin')

    device_col = f'device_{device_id}'
    device_data = df[df[device_col] == 1].sort_values('timestamp').reset_index(drop=True)

    mask = device_data['timestamp'] < start_ts
    if mask.sum() < SEQ_LEN:
        print(f"Error: Need {SEQ_LEN} rows before {start_ts}, found {mask.sum()}")
        return None

    seed = device_data[mask].tail(SEQ_LEN).reset_index(drop=True)

    # If single lat/lon given, auto-generate positions along the road
    if positions is None and mode == 'with_gps' and lat is not None:
        last_seed = seed.iloc[-1]
        # Build a fake last_row with the provided lat/lon and seed's speed/COG
        last_row_fake = {
            'Latitude': lat,
            'Longitude': lon,
            'speed_kmh': last_seed['speed_kmh'],
            'COG': last_seed['COG'],
            'timestamp': start_ts - pd.Timedelta(seconds=1),
        }
        
        positions = [{'lat': lat, 'lon': lon}]  # first step = given position
        for sec in range(1, n_steps):
            next_ts = start_ts + pd.Timedelta(seconds=sec)
            road_pos = get_road_position(str(next_ts), last_row_fake, net)
            if road_pos:
                positions.append({'lat': road_pos['lat'], 'lon': road_pos['lon']})
            else:
                positions.append({'lat': lat, 'lon': lon})  # fallback

    buffer = {col: list(seed[col].values) for col in ALL_AUTO_COLS}
    base_buffer = {col: list(seed[col].values) for col in BASE_INPUTS}

    generated_rows = []

    for step in range(n_steps):
        current_ts = start_ts + pd.Timedelta(seconds=step)

        current_base = {
            'hour': current_ts.hour,
            'day_of_week': current_ts.dayofweek + 1,
            'device_pc1': int(device_id == 'pc1'),
            'device_pc2': int(device_id == 'pc2'),
            'device_pc3': int(device_id == 'pc3'),
            'device_pc4': int(device_id == 'pc4'),
            #'direction_downlink': base_buffer['direction_downlink'][-1],
            'direction_uplink': base_buffer['direction_uplink'][-1],
            #'measured_qos_datarate': base_buffer['measured_qos_datarate'][-1],
            'measured_qos_delay': base_buffer['measured_qos_delay'][-1],
            'measurement': base_buffer['measurement'][-1],
            'operator': base_buffer['operator'][-1],
        }

        for col in ALL_AUTO_COLS:
            buffer[col].append(0.0)
        for col in BASE_INPUTS:
            base_buffer[col].append(current_base[col])

        base_seq = np.zeros((SEQ_LEN, len(BASE_INPUTS)), dtype=np.float32)
        auto_seq = np.zeros((SEQ_LEN, len(ALL_AUTO_COLS)), dtype=np.float32)

        for t in range(SEQ_LEN):
            idx = len(base_buffer['hour']) - SEQ_LEN + t
            for j, col in enumerate(BASE_INPUTS):
                base_seq[t, j] = base_buffer[col][idx]
            for j, col in enumerate(ALL_AUTO_COLS):
                auto_seq[t, j] = buffer[col][idx]

        auto_seq[-1, :] = 0.0

        step_lat = None
        step_lon = None
        if positions is not None and step < len(positions):
            step_lat = positions[step]['lat']
            step_lon = positions[step]['lon']

        if step_lat is not None:
            lat_idx = ALL_AUTO_COLS.index('Latitude')
            lon_idx = ALL_AUTO_COLS.index('Longitude')
            auto_seq[-1, lat_idx] = step_lat
            auto_seq[-1, lon_idx] = step_lon

        x = np.concatenate([base_seq, auto_seq], axis=1)
        x_scaled = input_scaler.transform(x.reshape(-1, x.shape[1])).reshape(1, SEQ_LEN, -1)

        with torch.no_grad():
            pred_scaled = model(torch.FloatTensor(x_scaled).to(device_torch)).cpu().numpy()
            pred = target_scaler.inverse_transform(pred_scaled)[0]

        result = {}
        for i, col in enumerate(ALL_TARGETS):
            val = pred[i]
            if col in SNAP_RULES:
                val = snap_to_nearest(val, SNAP_RULES[col])
            result[col] = val

        if step_lat is not None:
            result['Latitude'] = step_lat
            result['Longitude'] = step_lon

        result['COG'] = np.degrees(np.arctan2(result['sin_COG'], result['cos_COG'])) % 360
        result['jitter'] = np.expm1(result['jitter_log'])
        result['timestamp'] = current_ts

        for col in ALL_AUTO_COLS:
            buffer[col][-1] = result[col]

        for col in BASE_INPUTS:
            result[col] = current_base[col]

        generated_rows.append(result)

    return pd.DataFrame(generated_rows)

In [8]:
def format_output(gen_df):

    out = gen_df.copy()

    # Reverse log transforms
    LOG_FEATURES = [
        'datarate', 'target_datarate',
        'PCell_Downlink_TB_Size', 'PCell_Uplink_TB_Size',
        'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
        'PCell_Downlink_Num_RBs', 'PCell_Uplink_Num_RBs',
        'ping_ms', 'Pos in Ref Round', 'Traffic Distance',
        'PCell_Uplink_Tx_Power_(dBm)'
    ]

    for col in LOG_FEATURES:
        if col in out.columns:
            out[col] = np.expm1(out[col])

    # Decode one-hot: device
    if 'device_pc1' in out.columns:
        out['device'] = 'unknown'
        for d in ['pc1', 'pc2', 'pc3', 'pc4']:
            out.loc[out[f'device_{d}'] == 1, 'device'] = d
        out = out.drop(columns=['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4'], errors='ignore')
    
    # Decode one-hot: direction
    if 'direction_downlink' in out.columns:
        out['direction'] = 'unknown'
        #out.loc[out['direction_downlink'] == 1, 'direction'] = 'downlink'
        out.loc[out['direction_uplink'] == 1, 'direction'] = 'uplink'
        out = out.drop(columns=['direction_downlink', 'direction_uplink'], errors='ignore')

    # Decode one-hot: measured_qos
    if 'measured_qos_datarate' in out.columns:
        out['measured_qos'] = 'unknown'
        #out.loc[out['measured_qos_datarate'] == 1, 'measured_qos'] = 'datarate'
        out.loc[out['measured_qos_delay'] == 1, 'measured_qos'] = 'delay'
        out = out.drop(columns=['measured_qos_datarate', 'measured_qos_delay'], errors='ignore')

    # Drop internal columns
    drop_cols = ['sin_COG', 'cos_COG', 'jitter_log', 'hour', 'day_of_week']
    out = out.drop(columns=[c for c in drop_cols if c in out.columns], errors='ignore')

    # Round-off
    float_cols = out.select_dtypes(include=[np.number]).columns
    for col in float_cols:
        if col in ['Latitude', 'Longitude']:
            out[col] = out[col].round(7)
        elif col in ['PCell_freq_MHz', 'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                     'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                     'PCell_Downlink_Average_MCS', 'measurement', 'operator']:
            out[col] = out[col].round(0).astype(int)
        else:
            out[col] = out[col].round(2)

    first_cols = ['timestamp', 'device', 'direction', 'measured_qos']
    available_first = [c for c in first_cols if c in out.columns]
    remaining = [c for c in out.columns if c not in available_first]
    out = out[available_first + remaining]

    return out


In [9]:

# COMPARISON FUNCTION

def compare_with_real(device_id, start_timestamp, n_steps=5, mode='full',
                      lat=None, lon=None):
    
    
    start_ts = pd.Timestamp(start_timestamp)
    if start_ts.tzinfo is None:
        start_ts = start_ts.tz_localize('Europe/Berlin')

    device_col = f'device_{device_id}'
    device_data = df[df[device_col] == 1].sort_values('timestamp').reset_index(drop=True)

    real_mask = (device_data['timestamp'] >= start_ts) & \
                (device_data['timestamp'] < start_ts + pd.Timedelta(seconds=n_steps))
    real = device_data[real_mask].head(n_steps).reset_index(drop=True)

    gen = generate_direct_lstm(device_id, start_timestamp, n_steps, mode, lat, lon)
    if gen is None:
        return None, None

    n = min(len(gen), len(real))

    
    print(f" LSTM — {mode.upper()} | {device_id} | {start_timestamp} | {n_steps} steps")
    
    print(f"\n{'Feature':30s} | {'RMSE':>10s} | {'MAE':>10s} | {'Real (first 3)':>30s} | {'Gen (first 3)':>30s}")
    print("─" * 120)

    compare_cols = [
        ('Latitude', ''), ('Longitude', ''), ('speed_kmh', 'km/h'),
        ('COG', '°'), ('Altitude', 'm'), ('temperature', '°C'),
        ('PCell_RSRP_max', 'dBm'), ('PCell_freq_MHz', 'MHz'),
        ('PCell_Downlink_frequency', ''),
        ('datarate', '(log)'), ('ping_ms', '(log)'), ('jitter_log', '(log)'),
    ]

    for col, unit in compare_cols:
        if col not in real.columns or col not in gen.columns:
            continue
        rv = real[col].values[:n]
        gv = gen[col].values[:n]

        if col == 'COG':
            diff = np.abs(rv - gv)
            diff = np.minimum(diff, 360 - diff)
            rmse = np.sqrt(np.mean(diff ** 2))
            mae = np.mean(diff)
        else:
            rmse = np.sqrt(np.mean((rv - gv) ** 2))
            mae = np.mean(np.abs(rv - gv))

        r_str = ', '.join([f'{v:.4f}' for v in rv[:3]])
        g_str = ', '.join([f'{v:.4f}' for v in gv[:3]])
        print(f"{col + ' ' + unit:30s} | {rmse:10.4f} | {mae:10.4f} | {r_str:>30s} | {g_str:>30s}")

    return gen, real



In [10]:
# TEST

gen1, real1 = compare_with_real('pc2', '2021-06-23 14:12:51+02:00', n_steps=5,
                                 mode='with_gps', lat=52.504327, lon=13.335717)

 LSTM — WITH_GPS | pc2 | 2021-06-23 14:12:51+02:00 | 5 steps

Feature                        |       RMSE |        MAE |                 Real (first 3) |                  Gen (first 3)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Latitude                       |     0.0002 |     0.0002 |      52.5042, 52.5042, 52.5041 |      52.5043, 52.5043, 52.5043
Longitude                      |     0.0003 |     0.0003 |      13.3356, 13.3355, 13.3355 |      13.3357, 13.3358, 13.3358
speed_kmh km/h                 |     1.3639 |     1.1743 |         2.6755, 2.9128, 2.9809 |         2.4679, 2.2269, 1.8315
COG °                          |     2.2847 |     1.9098 |   206.3000, 206.6000, 206.7000 |   207.4446, 207.3112, 207.6991
Altitude m                     |     0.1846 |     0.1675 |      32.6000, 32.6000, 32.5000 |      32.8835, 32.8154, 32.5974
temperature °C                 |     0.1512 |     0.1388 |      20.4100, 20.410

In [11]:

# POINT VALIDATOR

def validate_point(lat, lon, speed, net=net):

    x, y = net.convertLonLat2XY(lon, lat)
    nearby = net.getNeighboringEdges(x, y, r=100)
    if not nearby:
        return {'on_road': False, 'snap_dist': 999, 'speed_ok': False,
                'speed_limit': 0, 'road': 'NO_ROAD'}
    valid = [(e, d) for e, d in nearby if not e.getID().startswith(':')]
    if not valid:
        valid = nearby
    nearest, dist = sorted(valid, key=lambda e: e[1])[0]
    sl = nearest.getSpeed() * 3.6
    return {
        'on_road': bool(dist <= 20),
        'snap_dist': round(float(dist), 1),
        'speed_ok': bool(speed <= sl * 1.1 + 5 or speed < 1),
        'speed_limit': round(sl, 1),
        'road': nearest.getName() or nearest.getID(),
    }

In [12]:

# POSITION ERROR

def position_error_m(lat1, lon1, lat2, lon2):
    """Haversine distance in meters."""
    R = 6_371_000
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat/2)**2 +
         np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2)
    return round(R * 2 * np.arcsin(np.sqrt(a)), 1)

In [13]:
def validate_beyond(segment, generate_fn, model_name, n_beyond=5,
                    device_id='pc1', net=net):
    last_row = segment.iloc[-1]
    last_ts = pd.Timestamp(last_row['timestamp'])
    
   
    print(f"  BEYOND-ENDPOINT VALIDATION: {model_name}")
    
    print(f"  Last timestamp: {last_ts}")
    print(f"  Last position:  {last_row['Latitude']:.6f}, {last_row['Longitude']:.6f}")
    print(f"  Last speed:     {last_row['speed_kmh']:.1f} km/h")
    print(f"  Last COG:       {last_row['COG']:.1f}°")
    
    beyond_points = []
    for sec in range(1, n_beyond + 1):
        new_ts = last_ts + pd.Timedelta(seconds=sec)
        road_pos = get_road_position(str(new_ts), last_row, net)
        if road_pos:
            beyond_points.append(road_pos)
    
    if not beyond_points:
        print("  Could not generate road positions")
        return None
    
    print(f"\n  Expected road positions:")
    for bp in beyond_points:
        print(f"    +{bp['time_diff']:.0f}s: {bp['lat']:.6f}, {bp['lon']:.6f} | "
              f"{bp['edge']} | {bp['dist_from_car']}m")
    
    # Generate ALL steps at once with positions
    start_ts_str = str(last_ts + pd.Timedelta(seconds=1))
    positions = [{'lat': bp['lat'], 'lon': bp['lon']} for bp in beyond_points]
    
    gen_df = generate_fn(device_id, start_ts_str, n_steps=n_beyond,
                         mode='with_gps', positions=positions)
    
    if gen_df is None or len(gen_df) == 0:
        print("  Generation failed")
        return None
    
    results = []
    for i, bp in enumerate(beyond_points):
        if i >= len(gen_df):
            break
        m1 = gen_df.iloc[i]
        m1v = validate_point(m1['Latitude'], m1['Longitude'], m1['speed_kmh'])
        row_result = {'road': bp, 'm1': m1.to_dict()}
        row_result['m1']['validation'] = m1v
        results.append(row_result)
    
    # Consistency check
    print(f"  CONSISTENCY: Last Real vs Generated")
    
    
    m1_first = results[0]['m1'] if results[0]['m1'] else None
    m1_last = results[-1]['m1'] if results[-1]['m1'] else None
    
    if m1_first and m1_last:
        features = [
            ('speed_kmh', 'Speed (km/h)'),
            ('PCell_RSRP_max', 'RSRP (dBm)'),
            ('PCell_RSRQ_max', 'RSRQ (dBm)'),
            ('PCell_RSSI_max', 'RSSI (dBm)'),
            ('PCell_SNR_1', 'SNR 1 (dB)'),
            ('PCell_SNR_2', 'SNR 2 (dB)'),
            ('PCell_freq_MHz', 'Frequency (MHz)'),
            ('PCell_Downlink_frequency', 'DL Frequency'),
            ('PCell_Band_Indicator', 'Band Indicator'),
            ('PCell_Downlink_bandwidth_MHz', 'DL BW (MHz)'),
            ('datarate', 'Datarate (log)'),
            ('ping_ms', 'Ping (log)'),
            ('jitter_log', 'Jitter (log)'),
            ('temperature', 'Temperature'),
        ]
        
        print(f"\n{'Feature':35s} | {'Last Real':>12s} | {'M1 +1s':>12s} | "
              f"{'M1 +{0}s'.format(n_beyond):>12s} | {'Diff +1s':>10s} | "
              f"{'Diff +{0}s'.format(n_beyond):>10s}")
        print("─" * 95)
        
        for col, label in features:
            if col in last_row.index and col in m1_first:
                real_val = float(last_row[col])
                val_1 = float(m1_first[col])
                val_n = float(m1_last[col])
                diff_1 = val_1 - real_val
                diff_n = val_n - real_val
                
                print(f"{label:35s} | {real_val:12.2f} | {val_1:12.2f} | "
                      f"{val_n:12.2f} | {diff_1:+10.2f} | {diff_n:+10.2f}")
    
    # Save to Excel
    last_5 = segment.tail(n_beyond).copy()
    last_5['source'] = 'REAL'
    
    gen_rows = []
    for entry in results:
        if entry['m1']:
            row = entry['m1'].copy()
            if 'validation' in row:
                row.pop('validation')
            row['source'] = 'GENERATED'
            gen_rows.append(row)
    
    gen_save = pd.DataFrame(gen_rows)
    
    common_cols = ['source'] + [c for c in last_5.columns if c in gen_save.columns and c != 'source']
    
    combined = pd.concat([
        last_5[common_cols].reset_index(drop=True),
        gen_save[common_cols].reset_index(drop=True)
    ], ignore_index=True)
    
    combined['timestamp'] = combined['timestamp'].apply(
        lambda x: str(x).split('+')[0] if '+' in str(x) else str(x)
    )
    
    excel_path = os.path.join(SUMO_DIR, f'beyond_validation_{model_name.replace(" ", "_")}_{device_id}.xlsx')
    combined.to_excel(excel_path, index=False, sheet_name='Beyond Validation')
    
    print(f"\n  Excel saved: {excel_path}")
    print(f"  Rows: {len(last_5)} real + {len(gen_rows)} generated = {len(combined)} total")
    print(f"  Columns: {len(combined.columns)}")
    
    return results

In [14]:
gen1=generate_direct_lstm('pc1', '2021-06-22 19:00:00+02:00', n_steps=5, mode='with_gps', lat=52.5134, lon=13.3341)
gen1

,Latitude,Longitude,speed_kmh,sin_COG,cos_COG,Altitude,precipIntensity,precipProbability,temperature,humidity,...,hour,day_of_week,device_pc1,device_pc2,device_pc3,device_pc4,direction_uplink,measured_qos_delay,measurement,operator
0,52.513400,13.334100,2.175680,0.677445,0.540301,34.177383,0.022623,0.010474,23.611542,0.577243,...,19,2,1,0,0,0,0,1,4,0
1,52.513341,13.334140,1.740185,0.706578,0.481864,34.227066,0.021667,0.010400,23.615162,0.577441,...,19,2,1,0,0,0,0,1,4,0
2,52.513342,13.334155,1.323622,0.728188,0.435539,34.191040,0.020515,0.010086,23.596909,0.578088,...,19,2,1,0,0,0,0,1,4,0
3,52.513342,13.334170,1.023162,0.748341,0.398153,34.166817,0.019759,0.009961,23.580040,0.578517,...,19,2,1,0,0,0,0,1,4,0
4,52.513343,13.334184,0.866131,0.762713,0.364694,34.230118,0.019402,0.009941,23.568943,0.578614,...,19,2,1,0,0,0,0,1,4,0


In [15]:
# Test 3: Formatted output
print("\n--- Formatted output ---")
if gen1 is not None:
    formatted = format_output(gen1)
    print(f"\nColumns ({len(formatted.columns)}):")
    print(list(formatted.columns))
    print(f"\n{formatted.to_string()}")




--- Formatted output ---

Columns (42):
['timestamp', 'device', 'Latitude', 'Longitude', 'speed_kmh', 'Altitude', 'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance', 'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max', 'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz', 'PCell_Downlink_frequency', 'PCell_Band_Indicator', 'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz', 'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size', 'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High', 'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)', 'datarate', 'Pos in Ref Round', 'target_datarate', 'ping_ms', 'COG', 'jitter', 'direction_uplink', 'measured_qos_delay', 'measurement', 'operator']

                  timestamp device   Latitude  Longitude  speed_kmh   Altitude  precipIntensity  precipProbability  temperature  humidity  windSpeed  Traffic Jam Fa

In [16]:
# First load dataset and setup
df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')
df = df.sort_values('timestamp').reset_index(drop=True)
df['sin_COG'] = np.sin(np.radians(df['COG']))
df['cos_COG'] = np.cos(np.radians(df['COG']))
df['jitter_log'] = np.log1p(df['jitter'])

pc1 = df[df['device_pc1'] == 1].sort_values('timestamp').reset_index(drop=True)



In [17]:
# Get all unique dates
dates = sorted(pc1['timestamp'].dt.date.unique())
print(f"Available dates: {dates}")

# Day 1
day1 = pc1[pc1['timestamp'].dt.date == dates[0]].reset_index(drop=True)

# Day 2
day2 = pc1[pc1['timestamp'].dt.date == dates[1]].reset_index(drop=True)

# Day 3
day3 = pc1[pc1['timestamp'].dt.date == dates[2]].reset_index(drop=True)

Available dates: [datetime.date(2021, 6, 22), datetime.date(2021, 6, 23), datetime.date(2021, 6, 24)]


In [18]:

#get_road_position — needs a timestamp and last_row

print("get_road_position")
last_row = day1.iloc[-1]  # last row of pc1
print(f"Last row: {last_row['timestamp']}, {last_row['Latitude']:.6f}, {last_row['Longitude']:.6f}, speed={last_row['speed_kmh']:.1f}")



get_road_position
Last row: 2021-06-22 18:13:59+02:00, 52.513543, 13.335668, speed=2.4


In [19]:
# timestamp = any timestamp
pos = get_road_position('2021-06-22 18:30:02+02:00', last_row)
print(f"Result: {pos}")

Result: {'lat': 52.516266308989266, 'lon': 13.332544643314826, 'edge': 'Englische Straße', 'speed_limit': 50.004000000000005, 'dist_from_car': 963.0, 'time_diff': 963.0, 'timestamp': '2021-06-22 18:30:02+02:00'}


In [20]:

# validate_point — needs lat, lon, speed

print("validate_point")

result = validate_point(52.5162, 13.3325, 20.0)
print(f"Result: {result}")


validate_point
Result: {'on_road': True, 'snap_dist': 0.7, 'speed_ok': True, 'speed_limit': 50.0, 'road': 'Englische Straße'}


In [21]:

print("validate_beyond")

gaps = day2['timestamp'].diff().dt.total_seconds()
break_points = gaps[gaps > 60].index.tolist()
starts = [0] + break_points
ends = break_points + [len(day2)]

# Pick last segment with moving car
segment = None
for s, e in reversed(list(zip(starts, ends))):
    seg = day2.iloc[s:e]
    if seg['speed_kmh'].iloc[-5:].mean() > 5 and len(seg) > 100:
        segment = seg.reset_index(drop=True)
        break

if segment is None:
    # Fallback: use last segment
    segment = day2.iloc[starts[-1]:ends[-1]].reset_index(drop=True)

print(f"Segment: {len(segment)} rows, {segment['timestamp'].iloc[0]} to {segment['timestamp'].iloc[-1]}")
print(f"End speed: {segment['speed_kmh'].iloc[-1]:.1f} km/h")

results = validate_beyond(segment, generate_direct_lstm, 'Direct LSTM', n_beyond=5)





validate_beyond
Segment: 3582 rows, 2021-06-23 16:07:16+02:00 to 2021-06-23 17:06:59+02:00
End speed: 3.7 km/h
  BEYOND-ENDPOINT VALIDATION: Direct LSTM
  Last timestamp: 2021-06-23 17:06:59+02:00
  Last position:  52.515667, 13.369463
  Last speed:     3.7 km/h
  Last COG:       89.5°

  Expected road positions:
    +1s: 52.515691, 13.369475 | Straße des 17. Juni | 1.0m
    +2s: 52.515691, 13.369490 | Straße des 17. Juni | 2.1m
    +3s: 52.515692, 13.369505 | Straße des 17. Juni | 3.1m
    +4s: 52.515693, 13.369520 | Straße des 17. Juni | 4.1m
    +5s: 52.515694, 13.369535 | Straße des 17. Juni | 5.1m
  CONSISTENCY: Last Real vs Generated

Feature                             |    Last Real |       M1 +1s |       M1 +5s |   Diff +1s |   Diff +5s
───────────────────────────────────────────────────────────────────────────────────────────────
Speed (km/h)                        |         3.69 |         3.74 |         3.64 |      +0.05 |      -0.05
RSRP (dBm)                          |    

In [22]:
def create_sumo_gui_beyond(segment, beyond_results, device_id='pc1'):
    device_data = df[df[f'device_{device_id}'] == 1].sort_values('timestamp').reset_index(drop=True)
    
    net_dir = os.path.dirname(net_file)
    add_file = os.path.join(SUMO_DIR, f'beyond_{device_id}.add.xml')
    route_file = os.path.join(net_dir, f'{device_id}.rou.xml')
    
    with open(add_file, 'w') as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write('<additional>\n')
        
        sampled = device_data.iloc[::5].reset_index(drop=True)
        print(f"Snapping {len(sampled)} GPS points to roads...")
        
        coords = []
        edge_ids = []
        for _, row in sampled.iterrows():
            x, y = net.convertLonLat2XY(row['Longitude'], row['Latitude'])
            nearby = net.getNeighboringEdges(x, y, r=50)
            if nearby:
                valid = [(e, d) for e, d in nearby if not e.getID().startswith(':')]
                if not valid:
                    valid = nearby
                edge, dist = sorted(valid, key=lambda e: e[1])[0]
                shape = edge.getShape()
                min_d = float('inf')
                snap_x, snap_y = x, y
                for i in range(len(shape) - 1):
                    x1, y1 = shape[i]
                    x2, y2 = shape[i + 1]
                    dx, dy = x2 - x1, y2 - y1
                    seg_len = np.sqrt(dx**2 + dy**2)
                    if seg_len > 0:
                        t = max(0, min(1, ((x - x1)*dx + (y - y1)*dy) / seg_len**2))
                        px, py = x1 + t*dx, y1 + t*dy
                        d = np.sqrt((x - px)**2 + (y - py)**2)
                        if d < min_d:
                            min_d = d
                            snap_x, snap_y = px, py
                coords.append(f"{snap_x:.2f},{snap_y:.2f}")
                edge_ids.append(edge.getID())
            else:
                coords.append(f"{x:.2f},{y:.2f}")
                edge_ids.append(None)
        
        gaps = device_data.iloc[::5]['timestamp'].diff().dt.total_seconds()
        break_indices = [0]
        for i, g in enumerate(gaps):
            if g and g > 120:
                break_indices.append(i)
        break_indices.append(len(coords))
        
        seg_count = 0
        for si in range(len(break_indices) - 1):
            start = break_indices[si]
            end = break_indices[si + 1]
            seg_coords = coords[start:end]
            if len(seg_coords) >= 2:
                f.write(f'    <poly id="route_seg{seg_count}" '
                        f'type="GPS trace segment {seg_count}" '
                        f'color="30,100,255,180" fill="0" layer="5" lineWidth="4" '
                        f'shape="{" ".join(seg_coords)}"/>\n')
                seg_count += 1
        
        print(f"Created {seg_count} trace segments")
        
        last_row = segment.iloc[-1]
        lx, ly = net.convertLonLat2XY(last_row['Longitude'], last_row['Latitude'])
        f.write(f'    <poi id="LAST_REAL" x="{lx:.2f}" y="{ly:.2f}" '
                f'color="255,255,255,255" '
                f'type="LAST REAL | {last_row["timestamp"]} | '
                f'Speed:{last_row["speed_kmh"]:.1f}kmh | '
                f'RSRP:{last_row["PCell_RSRP_max"]:.1f}dBm | '
                f'Freq:{last_row["PCell_freq_MHz"]:.0f}MHz" '
                f'layer="20"/>\n')
        
        line_coords = [f"{lx:.2f},{ly:.2f}"]
        
        for i, entry in enumerate(beyond_results):
            road = entry['road']
            gen = entry['m1']
            
            bx, by = net.convertLonLat2XY(road['lon'], road['lat'])
            line_coords.append(f"{bx:.2f},{by:.2f}")
            
            if gen:
                f.write(f'    <poi id="GEN_+{i+1}s" x="{bx:.2f}" y="{by:.2f}" '
                        f'color="255,140,0,255" '
                        f'type="GEN +{i+1}s | '
                        f'Speed:{gen["speed_kmh"]:.1f}kmh | '
                        f'RSRP:{gen["PCell_RSRP_max"]:.1f}dBm | '
                        f'Freq:{gen["PCell_freq_MHz"]:.0f}MHz | '
                        f'DR:{gen["datarate"]:.2f} | '
                        f'Ping:{gen["ping_ms"]:.2f}" '
                        f'layer="21"/>\n')
            else:
                f.write(f'    <poi id="ROAD_+{i+1}s" x="{bx:.2f}" y="{by:.2f}" '
                        f'color="255,165,0,255" '
                        f'type="ROAD +{i+1}s | {road["edge"]}" '
                        f'layer="21"/>\n')
        
        f.write(f'    <poly id="beyond_path" type="Generated beyond path" '
                f'color="255,140,0,200" fill="0" layer="6" lineWidth="5" '
                f'shape="{" ".join(line_coords)}"/>\n')
        
        f.write('</additional>\n')

    # --- Generate route file ---
    valid_edges = [eid for eid in edge_ids if eid is not None]
    # Deduplicate consecutive duplicate edges
    deduped_edges = [valid_edges[0]] if valid_edges else []
    for eid in valid_edges[1:]:
        if eid != deduped_edges[-1]:
            deduped_edges.append(eid)

    print(f"Writing route file with {len(deduped_edges)} edges...")
    with open(route_file, 'w') as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write('<routes>\n')
        f.write(f'    <vType id="{device_id}_type" vClass="passenger"/>\n')
        f.write(f'    <route id="{device_id}_route" edges="{" ".join(deduped_edges)}"/>\n')
        f.write(f'    <vehicle id="{device_id}" type="{device_id}_type" '
                f'route="{device_id}_route" depart="0"/>\n')
        f.write('</routes>\n')
    print(f"Route file written: {route_file}")

    # --- Generate SUMO config ---
    cfg_file = os.path.join(SUMO_DIR, f'beyond_{device_id}.sumocfg')
    
    # --- Generate SUMO config ---
    with open(cfg_file, 'w') as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write('<configuration>\n')
        f.write('    <input>\n')
        f.write(f'        <net-file value="{net_file}"/>\n')
        f.write(f'        <route-files value="{route_file}"/>\n')
        f.write(f'        <additional-files value="{add_file}"/>\n')
        f.write('    </input>\n')
        f.write('</configuration>\n')
    
    print(f"\n{'='*60}")
    print(f"SUMO GUI ready:")
    print(f"~/sumo/bin/sumo-gui -c {cfg_file}")
    print(f"{'='*60}")
    print(f"\nBlue lines: full {device_id} route (snapped to roads)")
    print(f"White dot:  last real position")
    print(f"Orange line + dots: generated points")
    print(f"Locate → POI → LAST_REAL to find beyond points")
    print(f"\nFiles created:")
    print(f"  {add_file}")
    print(f"  {route_file}")
    print(f"  {cfg_file}")
    

In [23]:
# SUMO GUI
create_sumo_gui_beyond(segment, results, device_id='pc1')

Snapping 11904 GPS points to roads...
Created 19 trace segments
Writing route file with 2711 edges...
Route file written: D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\pc1.rou.xml

SUMO GUI ready:
~/sumo/bin/sumo-gui -c D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\beyond_pc1.sumocfg

Blue lines: full pc1 route (snapped to roads)
White dot:  last real position
Orange line + dots: generated points
Locate → POI → LAST_REAL to find beyond points

Files created:
  D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\beyond_pc1.add.xml
  D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\pc1.rou.xml
  D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1\beyond_pc1.sumocfg
